In [1]:
import pandas as pd

# Load the dataset
data = pd.read_pickle('../../data/processed/cleaned_articles.pkl')

In [2]:
data.info()
data.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39863 entries, 0 to 39862
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   title              39863 non-null  object
 1   publisher          39863 non-null  object
 2   date               39863 non-null  object
 3   section            39863 non-null  object
 4   body               39863 non-null  object
 5   length             39863 non-null  int64 
 6   letters_flag       39863 non-null  bool  
 7   country_flag       39863 non-null  object
 8   foreign_countries  39863 non-null  object
 9   us_mentions        39863 non-null  object
 10  source_file        39863 non-null  object
 11  publisher_raw      39863 non-null  object
 12  section_clean      39845 non-null  object
 13  section_category   34176 non-null  object
dtypes: bool(1), int64(1), object(12)
memory usage: 4.0+ MB


,title,publisher,date,section,body,length,letters_flag,country_flag,foreign_countries,us_mentions,source_file,publisher_raw,section_clean,section_category
0,A Well-Documented Childhood. A Very Private Life.,New York Times,2024-12-31,Section A; Column 0; National Desk; Pg. 16,Jimmy Carter's daughter had an extraordinary a...,1209,False,BOTH,"[Canada, Egypt, Mexico, Nicaragua]","[Massachusetts, Rhode Island, US, Washington]",NYT/1.DOCX,The New York Times,section a; column 0; national desk; pg. 16,US/National
1,These were the big stories in arts and culture...,Other publisher,2024-12-31,WHAT TO KNOW,The arts in Dayton continued to thrive in 2024...,3345,False,BOTH,"[France, Japan, Paris (Capital), Sierra Leone,...","[America, California, Florida, Georgia, Massac...",Other publishers/Files (500) (1).DOCX,Dayton Daily News (Ohio),what to know,None
2,She Exalted The Beauty Of Dance,New York Times,2024-12-31,Section C; Column 0; The Arts/Cultural Desk; P...,She was The New Yorker's first dance critic. H...,1314,False,BOTH,[Male (Capital)],"[New York, North Carolina, US, United States]",NYT/1.DOCX,The New York Times,section c; column 0; the arts/cultural desk; p...,Arts/Culture
3,"Under a Highway in Rio, a Dance Style Charms a...",New York Times,2024-12-31,WORLD; americas,"Trucks, buses and cars rumbled overhead, drown...",1333,False,BOTH,"[Brazil, Lima (Capital)]","[New York, US, United States]",NYT/1.DOCX,The New York Times,world; americas,World/International
4,"Congressional pay, minimum wage stagnant for y...",Other publisher,2024-12-31,OPINION; Pg. A13,ABSTRACT\nMembers of Congress have not seen a ...,921,False,US_ONLY,[],"[America, Delaware, Maryland, New Jersey, New ...",Other publishers/Files (500) (1).DOCX,The Philadelphia Inquirer,opinion; pg. a13,Opinion/Editorial/Letters


In [3]:
print(data.columns)


Index(['title', 'publisher', 'date', 'section', 'body', 'length',
       'letters_flag', 'country_flag', 'foreign_countries', 'us_mentions',
       'source_file', 'publisher_raw', 'section_clean', 'section_category'],
      dtype='object')


### Preprocess with custom stopwords

In [4]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')

import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ---- 1. Define custom stopwords ----
custom_stopwords = {
    'said', 'say', 'saying', 'told',
    'mr', 'ms', 'mrs',
    'report', 'reported', 'interview', 'according',
    'article', 'photo', 'photograph', 'graphic',
    'someone', 'anyone', 'everyone', 'everybody',
    'york', 'new', 'time''much', 'many', 'way', 'often', 'even', 'always', 'got', 'get', 'never', 'among'

}

# ---- 2. Merge with NLTK stopwords ----
stop_words = set(stopwords.words('english')).union(custom_stopwords)

# ---- 3. Lemmatizer ----
lemmatizer = WordNetLemmatizer()

# ---- 4. Preprocess function ----
def preprocess(text):
    text = text.lower()  
    text = re.sub(r'[^a-z\s]', ' ', text)   # keep letters, spaces
    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]

    return ' '.join(words)

# ---- 5. Apply cleaning ----
data['clean_text'] = data['body'].astype(str).apply(preprocess)

data[['body', 'clean_text']].head()


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/jingguo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/jingguo/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,body,clean_text
0,Jimmy Carter's daughter had an extraordinary a...,jimmy carter daughter extraordinary well docum...
1,The arts in Dayton continued to thrive in 2024...,art dayton continued thrive exciting debut art...
2,She was The New Yorker's first dance critic. H...,yorker first dance critic wit could devastatin...
3,"Trucks, buses and cars rumbled overhead, drown...",truck bus car rumbled overhead drowning marcus...
4,ABSTRACT\nMembers of Congress have not seen a ...,abstract member congress seen pay raise since ...


### Vectorize (CountVectorizer)

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    max_features=50000,     # Limit vocabulary size for efficiency
    ngram_range=(1, 2)      # include unigrams + bigramslike "small business" and "self employment"

)

# Vectorize the clean_text column
X = vectorizer.fit_transform(data['clean_text'])

# Extract vocabulary
vocab = vectorizer.get_feature_names_out()

print("Vocabulary size:", len(vocab))
vocab[:30]   # Show the first 30 vocabulary items


Vocabulary size: 50000


array(['aa', 'aaa', 'aaron', 'aarp', 'aback', 'abaire', 'abandon',
       'abandoned', 'abandoned building', 'abandoned home',
       'abandoned house', 'abandoning', 'abandonment', 'abatement',
       'abbas', 'abbey', 'abbott', 'abbreviated', 'abby', 'abc',
       'abc news', 'abdel', 'abducted', 'abduction', 'abdul', 'abdullah',
       'abe', 'abel', 'aberration', 'abetted'], dtype=object)

### Define anchors + build anchor_indices + missing_anchors

In [6]:
# Step 3: Define anchor words 
anchor_words_dict = {
    'Housing': [
        'housing', 'rent', 'tenant', 'affordable', 'homelessness',
        'landlord', 'eviction', 'apartment', 'mortgage'
    ],
    'Education': [
        'education', 'school', 'college', 'teacher', 'university', 'tuition',
        'student', 'curriculum', 'classroom', 'learning'
    ],
    'Union': [
        'union', 'labor', 'strike', 'worker', 'collective',
        'organizing', 'wage', 'bargaining', 'employment'
    ],
    'Election': [
        'election', 'vote', 'campaign', 'party', 'candidate', 'rally',
        'politics', 'ballot', 'primary', 'poll', 'democrat', 'republican'
    ],
    'Military': [
        'military', 'service', 'veteran', 'army', 'soldier',
        'defense', 'troop', 'combat', 'deployment', 'navy'
    ],
    'Indebtedness': [
        'debt', 'struggle', 'credit', 'bankrupt', 'bankruptcy',
        'loan', 'foreclosure', 'repayment', 'interest', 'borrower', 'default', 'financial'
    ],
    'Family': [
        'family', 'child', 'parent', 'marriage', 'spouse', 'household',
        'domestic', 'raising', 'sibling', 'divorce', 'wedding', 'extended family', 'kin'  # bigram
    ],
    'Art': [
        'book', 'review', 'theater', 'play', 'movie',
        'culture', 'novel', 'film', 'gallery', 'music', 'performance', 'exhibition', 'artist'
    ],
    'Health': [
        'overdose', 'patient', 'disease', 'treatment', 'mental', 'therapy',
        'nurse', 'clinic', 'pandemic', 'loneliness', 'medicine', 'doctor'
    ],
    'Business Ownership': [
        'business', 'entrepreneur', 'self employment',   
        'client', 'customer', 'firm', 'owner', 'small business'   # bigram
    ],
    'Sports': [
        'soccer', 'hockey', 'athlete', 'league', 'tournament', 'game', 'score'
    ],
    'Religion': [
        'faith', 'islamic', 'sermon', 'worship', 'belief','religion', 'church', 'temple', 'mosque', 'christianity', 'catholic',
        'evangelical'
    ]
}

topic_names = list(anchor_words_dict.keys())

# Map words to vocabulary indices
word_to_idx = {w: i for i, w in enumerate(vocab)}

anchor_indices = []
for topic, words in anchor_words_dict.items():
    idx_list = [word_to_idx[w] for w in words if w in word_to_idx]
    anchor_indices.append(idx_list)
    print(f"{topic}: {len(idx_list)} anchors found out of {len(words)} total")


Housing: 9 anchors found out of 9 total
Education: 10 anchors found out of 10 total
Union: 9 anchors found out of 9 total
Election: 12 anchors found out of 12 total
Military: 10 anchors found out of 10 total
Indebtedness: 12 anchors found out of 12 total
Family: 13 anchors found out of 13 total
Art: 13 anchors found out of 13 total
Health: 12 anchors found out of 12 total
Business Ownership: 7 anchors found out of 8 total
Sports: 7 anchors found out of 7 total
Religion: 12 anchors found out of 12 total


In [7]:
# Inspect which anchor words are missing from the vocabulary
missing_anchors = {}

for topic, words in anchor_words_dict.items():
    missing = [w for w in words if w not in word_to_idx]
    if missing:
        missing_anchors[topic] = missing

missing_anchors


{'Business Ownership': ['self employment']}

### Train corex (strength=1) + print topics

In [8]:
from corextopic import corextopic as ct

corex = ct.Corex(
    n_hidden=len(topic_names),
    seed=42,
)

corex.fit(X, words=vocab, anchors=anchor_indices)


print("Total correlation (TC) explained:", corex.tc)


Total correlation (TC) explained: 128.38498074968277


In [9]:
for i, topic in enumerate(corex.get_topics(n_words=15)):
    print(f"\n### Topic {i+1}: {topic_names[i]}")
    for word, score, idx in topic:
        print(f"{word:20} {score:.3f}")



### Topic 1: Housing
city                 0.131
housing              0.124
resident             0.114
neighborhood         0.113
building             0.101
apartment            0.084
mayor                0.076
developer            0.075
community            0.075
property             0.073
rent                 0.069
development          0.068
real estate          0.066
brooklyn             0.064
avenue               0.064

### Topic 2: Education
school               0.130
student              0.117
education            0.094
college              0.085
university           0.084
high school          0.082
black                0.062
racial               0.058
graduate             0.052
teacher              0.052
study                0.049
grade                0.047
high                 0.045
public school        0.045
academic             0.042

### Topic 3: Union
policy               0.162
economic             0.109
government           0.095
leader               0.086
state           

### Train corex2 (strength=3) + print topic

In [10]:
from corextopic import corextopic as ct

# Create a new Corex model
corex2 = ct.Corex(
    n_hidden=len(topic_names),
    seed=42
)

# Fit with anchors and a stronger anchor_strength
corex2.fit(
    X,
    words=vocab,
    anchors=anchor_indices,
    anchor_strength=3 
)

print("TC:", corex2.tc)


TC: 137.52874897187465


In [11]:
for i, topic in enumerate(corex2.get_topics(n_words=15)):
    print(f"\n### Topic {i+1}: {topic_names[i]}")
    for word, score, idx in topic:
        print(f"{word:20} {score:.3f}")



### Topic 1: Housing
housing              0.446
apartment            0.293
rent                 0.247
tenant               0.191
affordable           0.155
city                 0.122
landlord             0.121
resident             0.109
neighborhood         0.108
developer            0.076
community            0.072
property             0.070
development          0.069
mayor                0.068
real estate          0.068

### Topic 2: Education
school               0.622
student              0.517
college              0.364
education            0.357
university           0.337
teacher              0.235
classroom            0.129
high school          0.111
tuition              0.100
learning             0.084
curriculum           0.076
graduate             0.062
high                 0.055
grade                0.052
public school        0.050

### Topic 3: Union
wage                 0.402
labor                0.353
worker               0.335
union                0.235
time opinion    

### Train corex2 (strength=5) + print topic

In [12]:
from corextopic import corextopic as ct

corex5 = ct.Corex(
    n_hidden=len(topic_names),
    seed=42
)

corex5.fit(
    X,
    words=vocab,
    anchors=anchor_indices,
    anchor_strength=5
)

print("Total correlation (TC):", corex5.tc)


Total correlation (TC): 147.6620553524576


In [13]:
n_words = 15  

for i, topic in enumerate(corex5.get_topics(n_words=n_words)):
    print(f"\n### Topic {i+1}: {topic_names[i]}")
    for word, score, idx in topic:
        print(f"{word:20} {score:.3f}")



### Topic 1: Housing
housing              0.678
apartment            0.571
rent                 0.426
tenant               0.304
affordable           0.224
landlord             0.201
city                 0.131
neighborhood         0.128
building             0.124
resident             0.119
mortgage             0.083
street               0.083
area                 0.081
property             0.074
park                 0.074

### Topic 2: Education
school               1.360
student              1.032
college              0.737
education            0.644
university           0.602
teacher              0.490
classroom            0.228
tuition              0.194
learning             0.147
curriculum           0.131
high school          0.128
graduate             0.060
high                 0.057
public school        0.052
grade                0.051

### Topic 3: Union
wage                 0.887
labor                0.836
worker               0.773
union                0.626
employment      

In [15]:
import pandas as pd

# ---------------------------------------------------
# 1. Put your topic names here (顺序必须和 anchor_indices 顺序一致)
# ---------------------------------------------------
topic_names = [
    "Housing",
    "Education",
    "Union",
    "Election",
    "Military",
    "Indebtedness",
    "Family",
    "Art",
    "Health",
    "Business Ownership",
    "Sports",
    "Religion"
]

# ---------------------------------------------------
# 2. Convert Corex output to readable string
# ---------------------------------------------------
def topic_to_string(topic):
    return "; ".join([f"{w} ({round(score,3)})" for w, score, idx in topic])


# ---------------------------------------------------
# 3. Build table combining strengths 1, 3, 5
# ---------------------------------------------------
rows = []

for i, name in enumerate(topic_names):

    row = {
        "Topic": name,
        "Top words (strength=1)": topic_to_string(corex.get_topics(n_words=15)[i]),
        "Top words (strength=3)": topic_to_string(corex2.get_topics(n_words=15)[i]),
        "Top words (strength=5)": topic_to_string(corex5.get_topics(n_words=15)[i]),
        "Anchor words": ", ".join(anchor_words_dict[name])
    }
    rows.append(row)

df = pd.DataFrame(rows)

# ---------------------------------------------------
# 4. Save as Excel
# ---------------------------------------------------
output_path = "corex_topics_comparison_new.xlsx"
df.to_excel(output_path, index=False)

output_path


'corex_topics_comparison_new.xlsx'